<a href="https://colab.research.google.com/github/gyan1131/Quantum_Computing_2026/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Qiskit Program with Readout Error Mitigation for Robust Output

Quantum computers are susceptible to various types of noise, which can corrupt the measurement results and make outputs less "robust." Readout error mitigation is a technique used to correct for systematic errors that occur during the measurement process. This example will demonstrate how to simulate a noisy measurement and then apply a basic readout error mitigator to improve the accuracy of the results.

In [ ]:
# Install Qiskit and related libraries if not already present
try:
    # Attempt to import all necessary Qiskit modules
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, pauli_error, depolarizing_error
    from qiskit.visualization import plot_histogram
    # NEW IMPORTS FOR MITIGATION IN QISKIT 1.0+
    from qiskit_experiments.library.characterization.measurement_calibration import MeasurementCalibrator
    from qiskit.result.mitigation import MeasurementFilter
    print("Qiskit and related components imported successfully (including new mitigation modules).")
    import qiskit_experiments # Add this to check if qiskit_experiments itself is loaded
    print(f"qiskit-experiments version: {qiskit_experiments.__version__}")

except ImportError as e_qiskit:
    print(f"Qiskit or required Qiskit components not found: {e_qiskit}, installing...")
    !pip install qiskit qiskit-aer qiskit-experiments # Added qiskit-experiments to install
    print("Qiskit, Qiskit-Aer, and Qiskit-Experiments installed. Attempting to re-import Qiskit modules...")
    try:
        from qiskit import QuantumCircuit, transpile
        from qiskit_aer import AerSimulator
        from qiskit_aer.noise import NoiseModel, pauli_error, depolarizing_error
        from qiskit.visualization import plot_histogram
        # NEW IMPORTS FOR MITIGATION IN QISKIT 1.0+
        from qiskit_experiments.library.characterization.measurement_calibration import MeasurementCalibrator
        from qiskit.result.mitigation import MeasurementFilter
        print("Qiskit modules re-imported successfully after installation (including new mitigation modules).")
        import qiskit_experiments # Add this to check if qiskit_experiments itself is loaded
        print(f"qiskit-experiments version: {qiskit_experiments.__version__}")
    except ImportError as e_reimport:
        print(f"Failed to import Qiskit modules even after installation: {e_reimport}. Please restart the runtime for changes to take effect.")
        raise # Re-raise the error as the core Qiskit imports are essential

# Separate imports for matplotlib and numpy as they are generally stable
try:
    import matplotlib.pyplot as plt
    import numpy as np
    print("matplotlib and numpy imported successfully.")
except ImportError as e_other:
    print(f"Failed to import matplotlib or numpy: {e_other}, installing...")
    !pip install matplotlib numpy
    try:
        import matplotlib.pyplot as plt
        import numpy as np
        print("matplotlib and numpy re-imported successfully after installation.")
    except ImportError as e_reimport_other:
        print(f"Failed to import matplotlib or numpy even after installation: {e_reimport_other}.")
        raise

Qiskit or required Qiskit components not found: No module named 'qiskit_experiments.library.characterization.measurement_calibration', installing...
Qiskit, Qiskit-Aer, and Qiskit-Experiments installed. Attempting to re-import Qiskit modules...
Failed to import Qiskit modules even after installation: No module named 'qiskit_experiments.library.characterization.measurement_calibration'. Please restart the runtime for changes to take effect.


ModuleNotFoundError: No module named 'qiskit_experiments.library.characterization.measurement_calibration'

In [ ]:
import qiskit
print(f"Qiskit Version: {qiskit.__version__}")

#### 1. Create a Simple Quantum Circuit

We'll create a basic circuit with a single qubit initialized in a superposition state using a Hadamard gate. Ideally, this should result in a 50/50 probability of measuring 0 or 1.

In [ ]:
# Create a Quantum Circuit with 1 qubit and 1 classical bit
qc = QuantumCircuit(1, 1)

# Apply a Hadamard gate to the qubit
qc.h(0)

# Measure the qubit
qc.measure(0, 0)

print("Quantum Circuit:")
print(qc)

#### 2. Set up a Basic Noise Model

We'll define a simple noise model for our simulator to mimic realistic quantum hardware behavior. This includes a readout error, where measurements might be flipped (0 to 1, or 1 to 0).

In [ ]:
# Define a custom noise model
noise_model = NoiseModel()

# Add a readout error to all qubits
# For qubit 0, 10% chance of 0 being read as 1, and 5% chance of 1 being read as 0
readout_error_matrix = [[0.9, 0.1], [0.05, 0.95]]
noise_model.add_readout_error(readout_error_matrix, [0])

# You can also add other errors like depolarizing error on gates if desired
# error_gate1 = depolarizing_error(0.05, 1) # 5% depolarizing error on single-qubit gates
# noise_model.add_all_qubit_quantum_error(error_gate1, ['h', 'x', 'y', 'z'])

print("Noise Model:")
print(noise_model)

#### 3. Simulate the Circuit with Noise

Now, we'll run our circuit on an `AerSimulator` configured with our custom noise model. Observe how the results deviate from the ideal 50/50 split due to the introduced readout error.

In [ ]:
# Select the AerSimulator with the custom noise model
simulator_noise = AerSimulator(noise_model=noise_model)

# Transpile the circuit for the simulator
compiled_circuit_noise = transpile(qc, simulator_noise)

# Run the noisy simulation
job_noise = simulator_noise.run(compiled_circuit_noise, shots=8192)

# Get the noisy result counts
noisy_counts = job_noise.result().get_counts(qc)

print("Noisy Measurement Counts:", noisy_counts)

# Plot noisy results
fig_noisy = plot_histogram(noisy_counts, title='Noisy Results', figsize=(6,4))
plt.show()

#### 4. Calibrate and Apply Readout Error Mitigator

To mitigate the readout errors, we first need to perform a calibration experiment. This involves running circuits that prepare qubits in known states (e.g., all 0s, all 1s) and measuring them. From these calibration results, a `TensoredReadoutMitigator` can be constructed and then applied to our noisy measurement results.

In [ ]:
# Generate the measurement calibration experiment
qubit_list = [0] # List of qubits to calibrate

# Create a MeasurementCalibrator experiment instance.
# This experiment creates the necessary calibration circuits and runs them.
# We pass the backend (simulator_noise) to the experiment constructor.
meas_calib_exp = MeasurementCalibrator(qubit_list, backend=simulator_noise, shots=8192)

# Run the calibration experiment on the simulator_noise backend.
# The simulator_noise already contains the noise_model, so it will run with noise.
calib_job = meas_calib_exp.run(simulator_noise)

# Wait for the job to complete and retrieve the experiment data.
calib_data = calib_job.block_until_done()

# Extract the calibration matrix (assignment matrix) from the analysis results.
# The MeasurementCalibrator's analysis will produce an AssignmentMatrix object.
calib_matrix = calib_data.analysis_results[0].value

# Create the MeasurementFilter from the calibration matrix.
# This object will be used to mitigate readout errors.
mitigator = MeasurementFilter(calib_matrix, qubit_list=qubit_list)

# Apply the mitigator to the noisy counts.
# The 'apply' method returns corrected counts (not quasi-probabilities directly).
mitigated_corrected_counts = mitigator.apply(noisy_counts)

# To match the original notebook's output style (binary probabilities),
# convert these corrected counts to probabilities.
total_shots = sum(noisy_counts.values())
mitigated_probabilities = {k: v / total_shots for k, v in mitigated_corrected_counts.items()}

print("Mitigated Measurement Probabilities:", mitigated_probabilities)
# Assign to 'mitigated_counts' to keep variable name consistent for later plotting cells.
mitigated_counts = mitigated_probabilities

#### 5. Compare Raw and Mitigated Results

Finally, let's visualize the raw noisy results against the mitigated (corrected) results to see the impact of error mitigation. You should observe that the mitigated results are closer to the ideal 50/50 distribution.

In [ ]:
# Convert mitigated quasi-probabilities to a format suitable for plot_histogram
# (plot_histogram expects integer counts, so we'll scale and round)
total_shots = sum(noisy_counts.values())
mitigated_display_counts = {k: int(v * total_shots) for k, v in mitigated_counts.items()}

# Plot both histograms side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_histogram(noisy_counts, ax=axes[0], title='Noisy Results (Raw)')
plot_histogram(mitigated_display_counts, ax=axes[1], title='Mitigated Results (Corrected)')

plt.tight_layout()
plt.show()

#### Constructing a Simple Quantum Circuit with a Hadamard Gate

This example demonstrates how to create a basic quantum circuit with a single qubit, apply a Hadamard gate to put it in a superposition state, and then measure it.

In [ ]:
# Create a Quantum Circuit with 1 qubit and 1 classical bit
qc_hadamard = QuantumCircuit(1, 1)

# Apply a Hadamard gate to the qubit at index 0
qc_hadamard.h(0)

# Measure the qubit and store the result in the classical bit at index 0
qc_hadamard.measure(0, 0)

print("Quantum Circuit with Hadamard Gate:")
print(qc_hadamard)

#### Simulating the Circuit with AerSimulator and Retrieving Results

This section demonstrates how to use Qiskit's `AerSimulator` to run a quantum circuit multiple times, collect the measurement outcomes, and display them. For a Hadamard gate on a single qubit, you would ideally expect a close to 50/50 split between '0' and '1'.

In [ ]:
import matplotlib.pyplot as plt

# Select the AerSimulator
simulator = AerSimulator()

# Transpile the circuit for the simulator
# This step is often necessary for optimization and ensures compatibility with the backend
compiled_qc_hadamard = transpile(qc_hadamard, simulator)

# Run the simulation with a specified number of shots (e.g., 1024)
# The 'shots' parameter determines how many times the circuit is executed
job = simulator.run(compiled_qc_hadamard, shots=1024)

# Get the results from the job
result = job.result()

# Get the measurement counts for the circuit
counts = result.get_counts(qc_hadamard)

print("Measurement Counts:", counts)

# Optionally, visualize the results as a histogram
fig = plot_histogram(counts, title='Hadamard Gate Measurement Results')
plt.show()

#### Apply Noise and Mitigation to `qc_hadamard`

Now, let's take the `qc_hadamard` circuit we just worked with, simulate it under the defined noise model, and then apply the readout error mitigator to see its effect.

In [ ]:
# Select the AerSimulator with the custom noise model
simulator_noise = AerSimulator(noise_model=noise_model)

# Transpile the qc_hadamard circuit for the simulator
compiled_qc_hadamard_noise = transpile(qc_hadamard, simulator_noise)

# Run the noisy simulation for qc_hadamard
job_qc_hadamard_noise = simulator_noise.run(compiled_qc_hadamard_noise, shots=8192)

# Get the noisy result counts for qc_hadamard
noisy_counts_qc_hadamard = job_qc_hadamard_noise.result().get_counts(qc_hadamard)

print("Noisy Measurement Counts for qc_hadamard:", noisy_counts_qc_hadamard)

# Plot noisy results for qc_hadamard
fig_noisy_qc_hadamard = plot_histogram(noisy_counts_qc_hadamard, title='Noisy Results (qc_hadamard)', figsize=(6,4))
plt.show()

In [ ]:
# Apply the existing mitigator to the noisy counts of qc_hadamard.
# The mitigator.apply method directly returns corrected counts.
mitigated_corrected_counts_qc_hadamard = mitigator.apply(noisy_counts_qc_hadamard)

# Convert corrected counts to probabilities to match the original output style.
total_shots_qc_hadamard = sum(noisy_counts_qc_hadamard.values())
mitigated_probabilities_qc_hadamard = {k: v / total_shots_qc_hadamard for k, v in mitigated_corrected_counts_qc_hadamard.items()}

print("Mitigated Measurement Probabilities for qc_hadamard:", mitigated_probabilities_qc_hadamard)
# Assign to 'mitigated_counts_qc_hadamard' to keep variable name consistent for later plotting cells.
mitigated_counts_qc_hadamard = mitigated_probabilities_qc_hadamard

In [ ]:
Ŵ# Convert mitigated quasi-probabilities to a format suitable for plot_histogram
# (plot_histogram expects integer counts, so we'll scale and round)
total_shots_qc_hadamard = sum(noisy_counts_qc_hadamard.values())
mitigated_display_counts_qc_hadamard = {k: int(v * total_shots_qc_hadamard) for k, v in mitigated_counts_qc_hadamard.items()}

# Plot both histograms side-by-side for qc_hadamard
fig_qc_hadamard, axes_qc_hadamard = plt.subplots(1, 2, figsize=(12, 5))

plot_histogram(noisy_counts_qc_hadamard, ax=axes_qc_hadamard[0], title='Noisy Results (qc_hadamard)')
plot_histogram(mitigated_display_counts_qc_hadamard, ax=axes_qc_hadamard[1], title='Mitigated Results (qc_hadamard)')

plt.tight_layout()
plt.show()

In [4]:
pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00


In [5]:
pip install matplotlib

In [6]:
pip install pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=063bfcfce9a583b58f47afa7effec2f841350089cb17cfb5a5085e476b71da81
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [7]:
pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.8 MB/s eta 0:00:00


In [14]:
import qiskit
print(qiskit.__version__)

2.4.1


In [15]:
counts = results[1].data.c.get_counts()
from qiskit.visualization import plot_histogram, plot_distribution plot_distribution(counts)

SyntaxError: invalid syntax (2707957323.py, line 2)

In [21]:
from qiskit_ibm_runtime import qiskitRuntimeService

ibm_service = QiskitRuntimeservice (
token = "Ohf4a29VBAbG2GcDZ9ZVMAzXTngxGAj_fWT",
instance = "crn:v1:bluemix:public:quantum-computing:us-east:a/8c82be223a29434aa10f15d83fc778c6:583e3d10-ccc4-4314-9f49-66c675f96f3d::" )

ImportError: cannot import name 'qiskitRuntimeService' from 'qiskit_ibm_runtime' (/usr/local/lib/python3.12/dist-packages/qiskit_ibm_runtime/__init__.py)